# Evidence-based Explainability for Brain-Tumour MRI
Companion to `tumor_enhanced.ipynb`. Every explanation here comes with **measured evidence**: where the model looked, *which features* the region has, whether the model **causally** depends on it, and how trustworthy the explanation itself is.

**Before running:** a trained model must exist in `models/`. Either run `python train.py --arch improved`, or add this to the end of your original notebook and run it:
```python
model.save("models/baseline_cnn.keras")
```

In [ ]:
import os, glob, json, numpy as np, matplotlib.pyplot as plt
from xai_brain.config import CATEGORIES, EvidenceConfig
from xai_brain.data import find_split_dir, list_images, load_images, load_image, sample_per_class, train_val_split
from xai_brain.model import load_model, make_predict_fn, get_embeddings, has_dropout, make_mc_dropout_fn
from xai_brain.saliency import build_explainers, parameter_randomization_test, lime_map, METHOD_LABELS
from xai_brain.channels import channel_evidence
from xai_brain.report import explain_image, make_evidence_figure, save_report
from xai_brain.uncertainty import fit_temperature, apply_temperature, expected_calibration_error
from xai_brain.surrogate import FeatureSurrogate, class_profiles
from xai_brain.evaluate import evaluate_dataset, summarize_evaluation
from xai_brain import global_analysis as ga

DATA = "archive"                       # folder containing Training/ and Testing/
MODEL_PATH = sorted(glob.glob("models/*.keras"))[-1]
print("using", MODEL_PATH)
model = load_model(MODEL_PATH); predict = make_predict_fn(model)
explainers, ge = build_explainers(model, predict, ("gradcam", "gradcampp", "scorecam", "ig", "occlusion"))
cfg = EvidenceConfig()
os.makedirs("outputs", exist_ok=True)

## 1. Is the confidence trustworthy? (calibration)
An explanation of a prediction is only useful if the quoted confidence means something. We fit a single *temperature* on the validation split and check the reliability diagram / ECE.

In [ ]:
paths, labels = list_images(find_split_dir(DATA, "Training"))
_, (va_p, va_y) = train_val_split(paths, labels, 0.1)
vp, vy = sample_per_class(va_p, va_y, 100)
val_probs = predict(load_images(vp))
T = fit_temperature(val_probs, vy)
before = expected_calibration_error(val_probs, vy); after = expected_calibration_error(apply_temperature(val_probs, T), vy)
print(f"temperature = {T:.2f} | ECE {before['ece']:.3f} -> {after['ece']:.3f}")
ga.plot_reliability(before, after); plt.show()

## 2. Pick a test image
`IDX` selects an image from a class-balanced sample. Set `ONLY_WRONG = True` to jump to a misclassified one (most instructive).

In [ ]:
te_p, te_y = list_images(find_split_dir(DATA, "Testing"))
sp, sy = sample_per_class(te_p, te_y, 25)
X = load_images(sp); probs = predict(X)
ONLY_WRONG, IDX = False, 0
if ONLY_WRONG:
    IDX = int(np.where(probs.argmax(1) != sy)[0][IDX])
img, true = X[IDX], int(sy[IDX])
print(sp[IDX], "| true:", CATEGORIES[true], "| predicted:", CATEGORIES[int(probs[IDX].argmax())])

## 3. The full evidence report for one image
One call produces: calibrated prediction + uncertainty (test-time augmentation, MC-dropout if available), 5 explanation methods with **faithfulness scores**, a consensus evidence region, **measured features of that region**, **necessity / sufficiency / specificity** counterfactual tests, internal feature-channel evidence, the built-in attention gate, a contrastive "why not class Y?", and a pass/fail evidence checklist.

In [ ]:
mc = make_mc_dropout_fn(model) if has_dropout(model) else None
res = explain_image(img, predict, explainers, CATEGORIES, true_label=true, config=cfg, temperature=T,
                    attention_fn=ge.attention if ge.gate_model else None,
                    channel_fn=lambda im, k: channel_evidence(ge, im, k, predict), mc_fn=mc)
fig = make_evidence_figure(res); plt.show()
print(res["text"])

### 3a. The checklist
Replaces the old single heuristic with individual, inspectable tests. Thresholds live in `EvidenceConfig`.

In [ ]:
import pandas as pd
pd.DataFrame(res["checklist"])[["name", "value", "rule", "passed"]]

### 3b. Which explanation method should you trust on this image?
Insertion/deletion AUC measures how well each map predicts what the model actually reacts to.

In [ ]:
pd.DataFrame({m["label"]: {"insertion AUC": m["auc_insertion"], "deletion AUC": m["auc_deletion"], "faithfulness": m["faithfulness"]}
              for m in res["methods"].values()}).T.sort_values("faithfulness", ascending=False)

### 3c. Features of the evidence region
These numbers are what the text explanation is built from.

In [ ]:
pd.Series(res["features"]).round(3).to_frame("value")

## 4. LIME (optional, slow) - superpixel evidence
Same API as the other methods, so it can also be added to `build_explainers(..., methods=(..., "lime"))`.

In [ ]:
m = lime_map(img, predict, res["explained_class"], num_samples=800)
plt.imshow(img); plt.imshow(m, cmap="jet", alpha=0.45); plt.axis("off"); plt.title("LIME (positive superpixels)"); plt.show()

## 5. Dataset-level evidence
Run the report on a balanced sample to answer: which method is most faithful on average? Do explanation metrics reveal when the model is wrong? Which region features characterise each class (surrogate model)?

In [ ]:
sanity = parameter_randomization_test(model, X[:10], probs[:10].argmax(1))
print("Randomisation sanity check passed:", sanity["passed"], {k: round(v, 2) for k, v in sanity.items() if k.startswith("mean")})
df, _ = evaluate_dataset(X, sy, predict, explainers, CATEGORIES, cfg, temperature=T, sanity=sanity)
df.to_csv("outputs/xai_metrics.csv", index=False)
summ = summarize_evaluation(df, CATEGORIES)
for k in ("method_faithfulness", "correct_vs_wrong", "error_detection_auroc", "per_class"):
    print("\n==", k); print(summ[k])
ga.plot_error_analysis(df); plt.show()

In [ ]:
prof = class_profiles(df, CATEGORIES)
sur = FeatureSurrogate(CATEGORIES).fit(df); g = sur.global_summary()
print(f"Surrogate agrees with the CNN {g['fidelity_logreg']:.0%} of the time (majority baseline {g['majority_baseline']:.0%})")
ga.plot_feature_importance(g["permutation_importance"]); plt.show()
print(g["tree_rules"])

In [ ]:
mm = ga.class_mean_maps(X, probs.argmax(1), explainers["gradcam"], CATEGORIES)
ga.plot_class_mean_maps(mm); plt.show()
ga.plot_tsne(get_embeddings(model, X), sy, CATEGORIES, probs.argmax(1)); plt.show()

## 6. Re-explain the chosen image with everything switched on
Now with class feature profiles, the surrogate attribution and nearest reference cases in the model's feature space.

In [ ]:
tr_p, tr_y = sample_per_class(paths, labels, 40)
nb = ga.make_neighbour_fn(lambda x: get_embeddings(model, x), get_embeddings(model, load_images(tr_p)), tr_y, tr_p)
res2 = explain_image(img, predict, explainers, CATEGORIES, true_label=true, config=cfg, temperature=T, profile=prof,
                     surrogate=sur, attention_fn=ge.attention if ge.gate_model else None, sanity=sanity,
                     channel_fn=lambda im, k: channel_evidence(ge, im, k, predict), neighbour_fn=nb, mc_fn=mc)
print(res2["text"])
save_report(res2, "outputs/case_report"); make_evidence_figure(res2); plt.show()